In [ ]:
import torch
import torchaudio
import json
import os
import random
from pathlib import Path
from stable_audio_tools.models.factory import create_model_from_config
from stable_audio_tools.models.utils import load_ckpt_state_dict
from stable_audio_tools.inference.generation import generate_diffusion_cond

# Generation

In [ ]:
def generate_samples(model, model_name, samples, class_mappings: str = "class_mappings.json", cfg:float  = 6.0, audio_length:int = 15, sample_rate = 44100, steps = 100, hybrid = True, device = "cuda"):

    with open(class_mappings, "r", encoding="utf-8") as file:
        class_mappings = json.load(file)

    sample_size = int(audio_length * sample_rate)

    base_dir = Path(f"generations/{model_name}")
    base_dir.mkdir(parents=True, exist_ok=True)

    for class_name, mapping in class_mappings.items():
        target_id = mapping["id"]
        target_str = mapping["name"]

        class_dir = base_dir / class_name
        class_dir.mkdir(exist_ok=True)

        if hybrid:
            prompt = f"A field recording of a {target_str} singing in nature, stereo audio."
        else:
            prompt = f"A field recording of a bird singing in nature, stereo audio."

        conditioning = [{
        "prompt": prompt,
        "seconds_start": 0,
        "seconds_total": audio_length,
        "species_id": target_id
        }]

        for sample_idx in range(samples):
            dynamic_seed = random.randint(0, 2147483647)
            with torch.no_grad():
                output = generate_diffusion_cond(
                    model,
                    steps=steps,
                    cfg_scale=cfg,
                    conditioning=conditioning,
                    sample_size=sample_size,
                    sigma_min=0.3,
                    sigma_max=500,
                    sampler_type="dpmpp-3m-sde",
                    device=device,
                    seed=dynamic_seed
                )

            file_out = class_dir / f"{target_str}_{sample_idx + 1}.wav"
            output = output.to(torch.float32).div(torch.max(torch.abs(output))).clamp(-1, 1).cpu()
            torchaudio.save(file_out, output, sample_rate)


In [7]:
CONFIG_PATH = "checkpoints/model_config_hybrid.json"
CKPT_PATH = "runs/hybrid_v1/models/epoch=22-step=6000.ckpt"

with open(CONFIG_PATH, "r") as f:
    model_config = json.load(f)

sample_rate = model_config["sample_rate"]
model = create_model_from_config(model_config)

In [ ]:
raw_state_dict = load_ckpt_state_dict(CKPT_PATH)

clean_state_dict = {}
for key, value in raw_state_dict.items():
    if key.startswith('diffusion_ema.ema_model.'):
        new_key = key.replace('diffusion_ema.ema_model.', '')
        clean_state_dict[new_key] = value

    elif key.startswith('diffusion.'):
        new_key = key.replace('diffusion.', '', 1)
        if new_key not in clean_state_dict:
            clean_state_dict[new_key] = value

model.load_state_dict(clean_state_dict, strict=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device, dtype=torch.float16)
model.eval()
print(device)

In [8]:
 # todo add generation calls

# Evaluation

In [ ]:
from frechet_audio_distance import FrechetAudioDistance

real_audio_dir = "/data_preprocessed_general"
generated_audio_dir ="/generations/model_name"

In [ ]:
def evaluate_quality(real_audio_dir, generated_audio_dir):
    frechet = FrechetAudioDistance(
        model_name="vggish",
        sample_rate=44100,
        use_pca=False,
        use_activation=False,
        verbose=True
    )
    try:
        fad_score = frechet.score(real_audio_dir, generated_audio_dir)
        print(f"\nFINAL FAD SCORE: {fad_score:.4f}")
    except Exception as e:
        print(f"FAD Calculation failed: {e}")